In [1]:
# nfl_data_py pulls from the nflverse data repository on GitHub, no API key needed
import nfl_data_py as nfl
import pandas as pd

# pulling one season first to inspect the raw data before committing to the full pull
pbp = nfl.import_pbp_data([2023])

print(pbp.shape)
print(pbp.columns.tolist())

2023 done.
Downcasting floats.
(49665, 396)
['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam', 'side_of_field', 'yardline_100', 'game_date', 'quarter_seconds_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'game_half', 'quarter_end', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'time', 'yrdln', 'ydstogo', 'ydsnet', 'desc', 'play_type', 'yards_gained', 'shotgun', 'no_huddle', 'qb_dropback', 'qb_kneel', 'qb_spike', 'qb_scramble', 'pass_length', 'pass_location', 'air_yards', 'yards_after_catch', 'run_location', 'run_gap', 'field_goal_result', 'kick_distance', 'extra_point_result', 'two_point_conv_result', 'home_timeouts_remaining', 'away_timeouts_remaining', 'timeout', 'timeout_team', 'td_team', 'td_player_name', 'td_player_id', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining', 'total_home_score', 'total_away_score', 'posteam_score', 'defteam_score', 'score_differential', 'posteam_score_post

In [2]:
# isolating the columns relevant to the EP model
# everything else is either an outcome, a player stat, or metadata
model_cols = [
    'play_id', 'game_id', 'down', 'ydstogo', 'yardline_100',
    'score_differential', 'half_seconds_remaining', 'qtr',
    'posteam_score', 'defteam_score', 'play_type', 'season_type'
]

subset = pbp[model_cols].copy()

print(subset.dtypes)
print("\n")
print(subset.isnull().sum())

play_id                   float32
game_id                    object
down                      float32
ydstogo                   float32
yardline_100              float32
score_differential        float32
half_seconds_remaining    float32
qtr                       float32
posteam_score             float32
defteam_score             float32
play_type                  object
season_type                object
dtype: object


play_id                      0
game_id                      0
down                      7740
ydstogo                      0
yardline_100              3460
score_differential        2638
half_seconds_remaining       5
qtr                          0
posteam_score             2638
defteam_score             2638
play_type                 1452
season_type                  0
dtype: int64


In [3]:
# filtering to scrimmage plays only, everything else has no EP context
# keeping only regular and postseason, no preseason
scrimmage = pbp[
    (pbp['play_type'].isin(['run', 'pass'])) &
    (pbp['season_type'].isin(['REG', 'POST'])) &
    (pbp['down'].notna()) &
    (pbp['yardline_100'].notna())
].copy()

print(scrimmage.shape)
print("\nnull counts after filter:")
print(scrimmage[model_cols].isnull().sum())

(35474, 396)

null counts after filter:
play_id                   0
game_id                   0
down                      0
ydstogo                   0
yardline_100              0
score_differential        0
half_seconds_remaining    0
qtr                       0
posteam_score             0
defteam_score             0
play_type                 0
season_type               0
dtype: int64


In [4]:
# pulling the full 2023 dataset again with scoring columns included
# we need posteam_score_post and defteam_score_post to track score changes
score_cols = [
    'play_id', 'game_id', 'qtr', 'down', 'ydstogo', 'yardline_100',
    'score_differential', 'half_seconds_remaining', 'game_seconds_remaining',
    'posteam', 'posteam_score', 'defteam_score',
    'posteam_score_post', 'defteam_score_post',
    'play_type', 'season_type', 'td_team', 'return_team',
    'total_home_score', 'total_away_score', 'home_team', 'away_team'
]

plays = scrimmage[score_cols].copy()

# sorting by game and play order so we can look forward correctly
plays = plays.sort_values(['game_id', 'play_id']).reset_index(drop=True)

print(plays.shape)
print(plays.head(3))


(35474, 22)
   play_id          game_id  qtr  down  ydstogo  yardline_100  \
0     55.0  2023_01_ARI_WAS  1.0   1.0     10.0          75.0   
1     77.0  2023_01_ARI_WAS  1.0   2.0      7.0          72.0   
2    102.0  2023_01_ARI_WAS  1.0   3.0      1.0          66.0   

   score_differential  half_seconds_remaining  game_seconds_remaining posteam  \
0                 0.0                  1800.0                  3600.0     WAS   
1                 0.0                  1770.0                  3570.0     WAS   
2                 0.0                  1735.0                  3535.0     WAS   

   ...  posteam_score_post  defteam_score_post  play_type  season_type  \
0  ...                 0.0                 0.0        run          REG   
1  ...                 0.0                 0.0       pass          REG   
2  ...                 0.0                 0.0        run          REG   

  td_team return_team total_home_score total_away_score  home_team  away_team  
0    None        None    

In [9]:
# resetting and trying a cleaner implementation
# the issue was the groupby apply was not assigning back correctly

next_score_list = []

# iterating through each game separately
for game_id, game_plays in plays.groupby('game_id'):
    events = game_plays['score_event'].values
    indices = game_plays.index.values
    
    for i in range(len(events)):
        found = False
        # look forward through remaining plays in this game
        for j in range(i + 1, len(events)):
            if events[j] is not None:
                next_score_list.append((indices[i], events[j]))
                found = True
                break
        if not found:
            next_score_list.append((indices[i], 'no_score'))

# converting to a series and mapping back to the dataframe
next_score_series = pd.Series(
    {idx: val for idx, val in next_score_list}
)
plays['next_score'] = next_score_series

print(plays['next_score'].value_counts())
print(f"\ntotal plays with next_score assigned: {plays['next_score'].notna().sum()}")

# inspecting the raw score changes to see what values are actually in the data
print("score_change value counts (nonzero only):")
print(plays[plays['score_change'] != 0]['score_change'].value_counts().head(20))

print("\nsample of plays where score changed:")
print(plays[plays['score_change'] != 0][['game_id', 'play_type', 'score_change', 'posteam_score', 'posteam_score_post', 'defteam_score', 'defteam_score_post']].head(10))

next_score
no_score      285
def_safety     15
Name: count, dtype: int64

total plays with next_score assigned: 300
score_change value counts (nonzero only):
score_change
 6.0    1296
-6.0      65
-2.0      15
Name: count, dtype: int64

sample of plays where score changed:
             game_id play_type  score_change  posteam_score  \
19   2023_01_ARI_WAS      pass           6.0            0.0   
56   2023_01_ARI_WAS      pass          -6.0            7.0   
96   2023_01_ARI_WAS       run           6.0           10.0   
166  2023_01_BUF_NYJ      pass           6.0            3.0   
222  2023_01_BUF_NYJ      pass           6.0            6.0   
264  2023_01_CAR_ATL      pass           6.0            0.0   
282  2023_01_CAR_ATL      pass           6.0            0.0   
323  2023_01_CAR_ATL       run           6.0           10.0   
341  2023_01_CAR_ATL       run           6.0           17.0   
426  2023_01_CIN_CLE       run           6.0            3.0   

     posteam_score_post  defteam

In [10]:
# nflfastR records TDs as 6 points, not 7
# the PAT is a separate play so we never see +7 in the raw score change
def classify_score(change):
    if change == 6:
        return 'off_td'
    elif change == -6:
        return 'def_td'
    elif change == 3:
        return 'off_fg'
    elif change == -3:
        return 'def_fg'
    elif change == 2:
        return 'off_safety'
    elif change == -2:
        return 'def_safety'
    else:
        return None

# reapplying with corrected values
plays['score_event'] = plays['score_change'].apply(classify_score)

print("scoring events found:")
print(plays['score_event'].value_counts())

scoring events found:
score_event
off_td        1296
def_td          65
def_safety      15
Name: count, dtype: int64


In [11]:
# checking what happens on field goal plays specifically
print("field goal result values:")
print(plays['field_goal_result'].value_counts() if 'field_goal_result' in plays.columns else "column not in subset")

# checking score change around known field goal plays
fg_plays = pbp[pbp['play_type'] == 'field_goal'][['game_id', 'play_id', 'play_type', 
    'posteam_score', 'posteam_score_post', 'defteam_score', 
    'defteam_score_post', 'field_goal_result']].head(10)
print(fg_plays)

field goal result values:
column not in subset
             game_id  play_id   play_type  posteam_score  posteam_score_post  \
36   2023_01_ARI_WAS    958.0  field_goal            0.0                 3.0   
46   2023_01_ARI_WAS   1237.0  field_goal            3.0                 6.0   
89   2023_01_ARI_WAS   2291.0  field_goal            7.0                10.0   
102  2023_01_ARI_WAS   2622.0  field_goal           13.0                16.0   
156  2023_01_ARI_WAS   4065.0  field_goal           17.0                20.0   
201  2023_01_BUF_NYJ    654.0  field_goal            0.0                 3.0   
219  2023_01_BUF_NYJ   1121.0  field_goal            0.0                 3.0   
248  2023_01_BUF_NYJ   1827.0  field_goal           10.0                13.0   
264  2023_01_BUF_NYJ   2216.0  field_goal            3.0                 6.0   
317  2023_01_BUF_NYJ   3518.0  field_goal           13.0                16.0   

     defteam_score  defteam_score_post field_goal_result  
36           

In [12]:
# pulling all plays from the full dataset to use for next score lookups
# we need field goals, punts, and special teams in this lookup
# even though we only model scrimmage plays
all_plays = pbp[pbp['season_type'].isin(['REG', 'POST'])].copy()
all_plays = all_plays.sort_values(['game_id', 'play_id']).reset_index(drop=True)

# recalculating score change on the full dataset
all_plays['score_change'] = (
    (all_plays['posteam_score_post'] - all_plays['posteam_score']) -
    (all_plays['defteam_score_post'] - all_plays['defteam_score'])
)

def classify_score(change):
    if change == 6:
        return 'off_td'
    elif change == -6:
        return 'def_td'
    elif change == 3:
        return 'off_fg'
    elif change == -3:
        return 'def_fg'
    elif change == 2:
        return 'off_safety'
    elif change == -2:
        return 'def_safety'
    else:
        return None

all_plays['score_event'] = all_plays['score_change'].apply(classify_score)

print("scoring events in full dataset:")
print(all_plays['score_event'].value_counts())

scoring events in full dataset:
score_event
off_td        1300
off_fg         950
def_td          77
off_safety      73
def_safety      19
Name: count, dtype: int64


In [13]:
# assigning next score using the full dataset so we never miss a scoring event
# that happened on a non-scrimmage play like a field goal or kickoff return
next_score_list = []

for game_id, game_plays in all_plays.groupby('game_id'):
    events = game_plays['score_event'].values
    indices = game_plays.index.values

    for i in range(len(events)):
        found = False
        for j in range(i + 1, len(events)):
            if events[j] is not None:
                next_score_list.append((indices[i], events[j]))
                found = True
                break
        if not found:
            next_score_list.append((indices[i], 'no_score'))

next_score_series = pd.Series({idx: val for idx, val in next_score_list})
all_plays['next_score'] = next_score_series

print("next score distribution across all plays:")
print(all_plays['next_score'].value_counts())

next score distribution across all plays:
next_score
off_td        1300
off_fg         950
no_score       285
def_td          77
off_safety      73
def_safety      19
Name: count, dtype: int64


In [14]:
# the previous approach only captured scoring plays themselves
# this version iterates every play and looks forward regardless of whether
# the current play is a scoring play or not

all_plays_sorted = all_plays.sort_values(['game_id', 'play_id']).reset_index(drop=True)

next_scores = []

games = all_plays_sorted['game_id'].unique()

for game in games:
    # get all plays for this game in order
    mask = all_plays_sorted['game_id'] == game
    game_idx = all_plays_sorted[mask].index.tolist()
    events = all_plays_sorted.loc[game_idx, 'score_event'].values

    for i in range(len(game_idx)):
        found = False
        for j in range(i + 1, len(events)):
            if events[j] is not None:
                next_scores.append((game_idx[i], events[j]))
                found = True
                break
        if not found:
            next_scores.append((game_idx[i], 'no_score'))

next_score_map = pd.Series({idx: val for idx, val in next_scores})
all_plays_sorted['next_score'] = next_score_map

print("next score distribution:")
print(all_plays_sorted['next_score'].value_counts())
print(f"\ntotal plays assigned: {all_plays_sorted['next_score'].notna().sum()}")
print(f"total plays in dataset: {len(all_plays_sorted)}")

next score distribution:
next_score
off_td        1300
off_fg         950
no_score       285
def_td          77
off_safety      73
def_safety      19
Name: count, dtype: int64

total plays assigned: 2704
total plays in dataset: 49665


In [15]:
# checking what the first game looks like to understand the index issue
game = games[0]
mask = all_plays_sorted['game_id'] == game
game_idx = all_plays_sorted[mask].index.tolist()
events = all_plays_sorted.loc[game_idx, 'score_event'].values

print(f"game: {game}")
print(f"number of plays: {len(game_idx)}")
print(f"first 5 indices: {game_idx[:5]}")
print(f"first 5 events: {events[:5]}")
print(f"non-null events: {sum(e is not None for e in events)}")

game: 2023_01_ARI_WAS
number of plays: 175
first 5 indices: [0, 1, 2, 3, 4]
first 5 events: <StringArray>
[nan, nan, nan, nan, nan]
Length: 5, dtype: str
non-null events: 175


In [20]:
# the fix is checking for the string 'nan' and actual NaN values
# pandas StringArray stores missing as pd.NA not None or float nan

import pandas as pd

# rewriting classify_score to return pd.NA instead of None
def classify_score(change):
    if change == 6:
        return 'off_td'
    elif change == -6:
        return 'def_td'
    elif change == 3:
        return 'off_fg'
    elif change == -3:
        return 'def_fg'
    elif change == 2:
        return 'off_safety'
    elif change == -2:
        return 'def_safety'
    else:
        return pd.NA

all_plays_sorted['score_event'] = all_plays_sorted['score_change'].apply(classify_score)

# verifying the dtype is now object not StringArray
print(all_plays_sorted['score_event'].dtype)
print(all_plays_sorted['score_event'].value_counts())
print(f"\nsample null check: {all_plays_sorted['score_event'].iloc[0]}")
print(f"is na: {pd.isna(all_plays_sorted['score_event'].iloc[0])}")

str
score_event
off_td        1300
off_fg         950
def_td          77
off_safety      73
def_safety      19
Name: count, dtype: int64

sample null check: nan
is na: True


In [21]:
# rerunning the next score assignment with the corrected null check
next_scores = []

for game in games:
    mask = all_plays_sorted['game_id'] == game
    game_idx = all_plays_sorted[mask].index.tolist()
    events = all_plays_sorted.loc[game_idx, 'score_event'].values

    for i in range(len(game_idx)):
        found = False
        for j in range(i + 1, len(events)):
            # using pd.isna instead of None check since dtype is str with nan
            if not pd.isna(events[j]):
                next_scores.append((game_idx[i], events[j]))
                found = True
                break
        if not found:
            next_scores.append((game_idx[i], 'no_score'))

next_score_map = pd.Series({idx: val for idx, val in next_scores})
all_plays_sorted['next_score'] = next_score_map

print("next score distribution:")
print(all_plays_sorted['next_score'].value_counts())
print(f"\ntotal plays assigned: {all_plays_sorted['next_score'].notna().sum()}")
print(f"total plays in dataset: {len(all_plays_sorted)}")

next score distribution:
next_score
off_td        23629
off_fg        19134
no_score       5284
def_td         1193
def_safety      339
off_safety       86
Name: count, dtype: int64

total plays assigned: 49665
total plays in dataset: 49665


In [22]:
# joining next_score back to scrimmage plays only
# we built next_score on the full dataset so field goals and
# special teams plays were visible during the lookup
plays = plays.join(all_plays_sorted[['next_score']], how='left')

print("next score distribution on scrimmage plays only:")
print(plays['next_score'].value_counts())
print(f"\ntotal scrimmage plays: {len(plays)}")
print(f"plays with next_score assigned: {plays['next_score'].notna().sum()}")

ValueError: columns overlap but no suffix specified: Index(['next_score'], dtype='str')

In [23]:
# dropping the previous next_score column from the earlier failed attempt
plays = plays.drop(columns=['next_score'], errors='ignore')

# joining next_score from the full dataset lookup
plays = plays.join(all_plays_sorted[['next_score']], how='left')

print("next score distribution on scrimmage plays only:")
print(plays['next_score'].value_counts())
print(f"\ntotal scrimmage plays: {len(plays)}")
print(f"plays with next_score assigned: {plays['next_score'].notna().sum()}")

next score distribution on scrimmage plays only:
next_score
off_td        16715
off_fg        13722
no_score       3936
def_td          803
def_safety      234
off_safety       64
Name: count, dtype: int64

total scrimmage plays: 35474
plays with next_score assigned: 35474


In [24]:
# pulling all three seasons at once
# 2021-2022 will be training data, 2023 will be held out for validation
pbp_all = nfl.import_pbp_data([2021, 2022, 2023])

print(pbp_all.shape)
print(pbp_all['season'].value_counts().sort_index())

2021 done.
2022 done.
2023 done.
Downcasting floats.
(149021, 396)
season
2021    49922
2022    49434
2023    49665
Name: count, dtype: int64


In [25]:
# filtering to scrimmage plays across all three seasons
scrimmage_all = pbp_all[
    (pbp_all['play_type'].isin(['run', 'pass'])) &
    (pbp_all['season_type'].isin(['REG', 'POST'])) &
    (pbp_all['down'].notna()) &
    (pbp_all['yardline_100'].notna())
].copy()

print(f"scrimmage plays across all seasons: {scrimmage_all.shape}")
print(scrimmage_all['season'].value_counts().sort_index())

scrimmage plays across all seasons: (106386, 396)
season
2021    35608
2022    35304
2023    35474
Name: count, dtype: int64


In [26]:
# running the full next score derivation on all three seasons
# same logic as before, just on the full dataset
all_plays_full = pbp_all[pbp_all['season_type'].isin(['REG', 'POST'])].copy()
all_plays_full = all_plays_full.sort_values(['game_id', 'play_id']).reset_index(drop=True)

# score change from the offensive team's perspective
all_plays_full['score_change'] = (
    (all_plays_full['posteam_score_post'] - all_plays_full['posteam_score']) -
    (all_plays_full['defteam_score_post'] - all_plays_full['defteam_score'])
)

# classifying scoring events
all_plays_full['score_event'] = all_plays_full['score_change'].apply(classify_score)

# assigning next score by looking forward within each game
next_scores_full = []
games_full = all_plays_full['game_id'].unique()

for game in games_full:
    mask = all_plays_full['game_id'] == game
    game_idx = all_plays_full[mask].index.tolist()
    events = all_plays_full.loc[game_idx, 'score_event'].values

    for i in range(len(game_idx)):
        found = False
        for j in range(i + 1, len(events)):
            if not pd.isna(events[j]):
                next_scores_full.append((game_idx[i], events[j]))
                found = True
                break
        if not found:
            next_scores_full.append((game_idx[i], 'no_score'))

next_score_map_full = pd.Series({idx: val for idx, val in next_scores_full})
all_plays_full['next_score'] = next_score_map_full

print("next score distribution across all three seasons:")
print(all_plays_full['next_score'].value_counts())
print(f"\ntotal plays assigned: {all_plays_full['next_score'].notna().sum()}")

next score distribution across all three seasons:
next_score
off_td        73225
off_fg        56466
no_score      15125
def_td         3210
def_safety      722
off_safety      273
Name: count, dtype: int64

total plays assigned: 149021


In [27]:
# joining next_score back to scrimmage plays across all three seasons
scrimmage_all = scrimmage_all.drop(columns=['next_score'], errors='ignore')
scrimmage_all = scrimmage_all.join(all_plays_full[['next_score']], how='left')

print("next score distribution on scrimmage plays:")
print(scrimmage_all['next_score'].value_counts())
print(f"\ntotal scrimmage plays: {len(scrimmage_all)}")
print(f"plays with next_score assigned: {scrimmage_all['next_score'].notna().sum()}")

next score distribution on scrimmage plays:
next_score
off_td        53418
off_fg        41023
no_score       9057
def_td         2173
def_safety      497
off_safety      218
Name: count, dtype: int64

total scrimmage plays: 106386
plays with next_score assigned: 106386


In [29]:
import os

os.makedirs('data', exist_ok=True)

scrimmage_all.to_csv('data/scrimmage_2021_2023.csv', index=True)

print("saved to data/scrimmage_2021_2023.csv")
print(f"\nshape: {scrimmage_all.shape}")
print(scrimmage_all[['season', 'down', 'ydstogo', 'yardline_100', 
                      'score_differential', 'half_seconds_remaining', 
                      'next_score']].head(5))

saved to data/scrimmage_2021_2023.csv

shape: (106386, 397)
   season  down  ydstogo  yardline_100  score_differential  \
2    2021   1.0     10.0          75.0                 0.0   
3    2021   2.0     13.0          78.0                 0.0   
4    2021   3.0     10.0          75.0                 0.0   
6    2021   1.0     10.0          61.0                 0.0   
7    2021   1.0     10.0          23.0                 0.0   

   half_seconds_remaining next_score  
2                  1800.0     off_fg  
3                  1763.0     off_fg  
4                  1722.0     off_fg  
6                  1707.0     off_fg  
7                  1675.0     off_fg  
